# Day 2 — Hybrid Retrieval + Reranking

Goal for today: build each piece of the retrieval stage of the RAG pipeline by hand, in isolation, before wiring it into LangGraph later.

Pipeline for today (no LLM calls yet):

```
toy corpus -> BM25 retriever -> 
            -> Dense (FAISS) retriever -> EnsembleRetriever (hybrid fusion) -> Cross-encoder reranker
```

We'll build one box at a time, run it, and actually look at the output before adding the next box.

## Step 1: Environment check + toy corpus

The corpus below is deliberately small and deliberately mixed:
- A few docs are keyed around **exact identifiers** (SKUs) — the kind of thing BM25 is good at and embeddings are bad at.
- A few docs describe the **same concepts in different words** — the kind of thing embeddings are good at and BM25 is bad at.

This lets us actually *see* the two retrievers disagree later, instead of taking it on faith.

In [21]:
# Sanity check on the core packages we'll use today.
# If any of these fail, install with: pip install <package> --break-system-packages
import langchain
import langchain_community
print("langchain:", langchain.__version__)
print("langchain_community:", langchain_community.__version__)

langchain: 0.3.27
langchain_community: 0.3.19


In [22]:
from langchain_core.documents import Document

# Toy corpus: retail-style docs mixing exact identifiers and paraphrased descriptions.
# doc_id is just for our own tracking in this notebook, not a LangChain concept.
raw_docs = [
    {"doc_id": "d1", "text": "SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund."},
    {"doc_id": "d2", "text": "If you'd like to send back something you bought, you generally have a month from the purchase date, as long as you kept your proof of purchase."},
    {"doc_id": "d3", "text": "SKU-77310 is a stainless steel water bottle, 32oz capacity, dishwasher safe."},
    {"doc_id": "d4", "text": "Our insulated bottles keep drinks cold for up to 24 hours and are safe to clean in a dishwasher."},
    {"doc_id": "d5", "text": "Store credit is issued instead of a refund for returns made after 30 days but within 90 days."},
    {"doc_id": "d6", "text": "SKU-90112 wireless earbuds come with a 1-year manufacturer warranty covering defects."},
    {"doc_id": "d7", "text": "Electronics purchased from us are covered against manufacturing defects for twelve months from purchase."},
    {"doc_id": "d8", "text": "Gift cards cannot be returned or exchanged for cash under any circumstances."},
    {"doc_id": "d9", "text": "SKU-88421 is currently out of stock online but available for in-store pickup at select locations."},
    {"doc_id": "d10", "text": "Curbside pickup orders must be collected within 5 days or they will be restocked automatically."},
]

documents = [
    Document(page_content=d["text"], metadata={"doc_id": d["doc_id"]})
    for d in raw_docs
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  [{doc.metadata['doc_id']}] {doc.page_content}")

Loaded 10 documents
  [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  [d2] If you'd like to send back something you bought, you generally have a month from the purchase date, as long as you kept your proof of purchase.
  [d3] SKU-77310 is a stainless steel water bottle, 32oz capacity, dishwasher safe.
  [d4] Our insulated bottles keep drinks cold for up to 24 hours and are safe to clean in a dishwasher.
  [d5] Store credit is issued instead of a refund for returns made after 30 days but within 90 days.
  [d6] SKU-90112 wireless earbuds come with a 1-year manufacturer warranty covering defects.
  [d7] Electronics purchased from us are covered against manufacturing defects for twelve months from purchase.
  [d8] Gift cards cannot be returned or exchanged for cash under any circumstances.
  [d9] SKU-88421 is currently out of stock online but available for in-store pickup at select locations.
  [d10] Curbside pickup orders must 

## Step 2: BM25 retriever, in isolation

`BM25Retriever` from `langchain_community` builds a lexical index over the documents (no embeddings, no model calls). We'll query it with:
1. An **exact-identifier query** (`SKU-88421`) — BM25's home turf.
2. A **paraphrased query** with no shared vocabulary with the source doc — BM25's weak spot.

Before running this: which doc(s) would you predict come back for each query, based on the corpus above?

In [23]:
# If this import fails: pip install rank_bm25 --break-system-packages
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents, k=3)

def show_results(retriever, query, label):
    print(f"--- {label} ---")
    print(f"Query: {query!r}\n")
    results = retriever.invoke(query)
    for rank, doc in enumerate(results, start=1):
        print(f"  #{rank} [{doc.metadata['doc_id']}] {doc.page_content}")
    print()
    return results

In [24]:
# Query 1: exact identifier match — BM25's strength
_ = show_results(bm25_retriever, "SKU-88421 availability", "BM25: exact SKU query")

# Query 2: paraphrased, no shared vocabulary with d2 — BM25's weak spot
_ = show_results(bm25_retriever, "how long do I have to return something", "BM25: paraphrased query")

--- BM25: exact SKU query ---
Query: 'SKU-88421 availability'

  #1 [d9] SKU-88421 is currently out of stock online but available for in-store pickup at select locations.
  #2 [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #3 [d10] Curbside pickup orders must be collected within 5 days or they will be restocked automatically.

--- BM25: paraphrased query ---
Query: 'how long do I have to return something'

  #1 [d2] If you'd like to send back something you bought, you generally have a month from the purchase date, as long as you kept your proof of purchase.
  #2 [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #3 [d4] Our insulated bottles keep drinks cold for up to 24 hours and are safe to clean in a dishwasher.



## Step 3: Dense retriever (FAISS + sentence-transformers), in isolation

Same corpus, same two queries — but now via embeddings. We're using `all-MiniLM-L6-v2`: small, fast, and fine for local experimentation (not what you'd necessarily run in production at 10M+ docs, but the mechanics are identical to whatever embedding model you eventually pick).

Prediction check again: for the paraphrase query, do you expect `d2` to show up — and this time, do you expect it to show up *because* the model understands the meaning, or by accident like BM25's case?

In [25]:
# If these imports fail:
#   pip install sentence-transformers faiss-cpu --break-system-packages
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8042.39it/s]


In [26]:
_ = show_results(dense_retriever, "SKU-88421 availability", "Dense: exact SKU query")
_ = show_results(dense_retriever, "how long do I have to return something", "Dense: paraphrased query")

--- Dense: exact SKU query ---
Query: 'SKU-88421 availability'

  #1 [d9] SKU-88421 is currently out of stock online but available for in-store pickup at select locations.
  #2 [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #3 [d3] SKU-77310 is a stainless steel water bottle, 32oz capacity, dishwasher safe.

--- Dense: paraphrased query ---
Query: 'how long do I have to return something'

  #1 [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #2 [d2] If you'd like to send back something you bought, you generally have a month from the purchase date, as long as you kept your proof of purchase.
  #3 [d5] Store credit is issued instead of a refund for returns made after 30 days but within 90 days.



## Step 4: Hybrid fusion with `EnsembleRetriever`

`EnsembleRetriever` (from core `langchain`, not `langchain_community`) takes a list of retrievers plus a matching list of weights, runs each retriever independently, and fuses the ranked lists via **Reciprocal Rank Fusion** — a doc's fused score depends on *what rank it held in each list*, not the raw BM25/cosine scores (which aren't on comparable scales to begin with).

Weights of `[0.5, 0.5]` below mean equal trust in both retrievers for now — in a real system you'd tune this based on how identifier-heavy vs. conceptual your actual queries tend to be.

Prediction before running: for the SKU query, do you expect fusion to demote `d3` (the wrong SKU) compared to dense-alone? For the paraphrase query, do you expect BM25's accidental `d4` match to survive fusion?

In [27]:
from langchain.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

In [28]:
_ = show_results(ensemble_retriever, "SKU-88421 availability", "Ensemble: exact SKU query")
_ = show_results(ensemble_retriever, "how long do I have to return something", "Ensemble: paraphrased query")

--- Ensemble: exact SKU query ---
Query: 'SKU-88421 availability'

  #1 [d9] SKU-88421 is currently out of stock online but available for in-store pickup at select locations.
  #2 [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #3 [d10] Curbside pickup orders must be collected within 5 days or they will be restocked automatically.
  #4 [d3] SKU-77310 is a stainless steel water bottle, 32oz capacity, dishwasher safe.

--- Ensemble: paraphrased query ---
Query: 'how long do I have to return something'

  #1 [d2] If you'd like to send back something you bought, you generally have a month from the purchase date, as long as you kept your proof of purchase.
  #2 [d1] SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #3 [d4] Our insulated bottles keep drinks cold for up to 24 hours and are safe to clean in a dishwasher.
  #4 [d5] Store credit is issued instead of a refund for r

## Step 5: Cross-encoder reranking

Unlike BM25/dense retrieval (which pre-score each doc independently of the query, or independently of each other), a cross-encoder takes the **query and one candidate document together** and outputs a single relevance score for that pair. This is why it can't be used to search 10M+ docs directly — it has to run once per candidate, at request time — but run over a small candidate set (the output of fusion), it's cheap enough and far more accurate.

We're using `cross-encoder/ms-marco-MiniLM-L-6-v2`, a small, widely-used reranker trained on real search relevance judgments (MS MARCO).

Prediction: given what just happened with `d4` vs. `d5` in fusion, do you expect the reranker to fix that ordering — and why would it succeed where rank fusion couldn't?

In [29]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidate_docs, top_k=None):
    pairs = [(query, doc.page_content) for doc in candidate_docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidate_docs, scores), key=lambda x: x[1], reverse=True)
    if top_k:
        ranked = ranked[:top_k]
    print(f"Query: {query!r}\n")
    for rank, (doc, score) in enumerate(ranked, start=1):
        print(f"  #{rank} [{doc.metadata['doc_id']}] score={score:.3f}  {doc.page_content}")
    print()
    return ranked

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16733.23it/s]


In [30]:
sku_query = "SKU-88421 availability"
paraphrase_query = "how long do I have to return something"

sku_candidates = ensemble_retriever.invoke(sku_query)
paraphrase_candidates = ensemble_retriever.invoke(paraphrase_query)

print("=== Reranked: SKU query ===")
_ = rerank(sku_query, sku_candidates)

print("=== Reranked: paraphrase query ===")
_ = rerank(paraphrase_query, paraphrase_candidates)

=== Reranked: SKU query ===
Query: 'SKU-88421 availability'

  #1 [d9] score=6.925  SKU-88421 is currently out of stock online but available for in-store pickup at select locations.
  #2 [d1] score=4.578  SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #3 [d3] score=0.805  SKU-77310 is a stainless steel water bottle, 32oz capacity, dishwasher safe.
  #4 [d10] score=-11.254  Curbside pickup orders must be collected within 5 days or they will be restocked automatically.

=== Reranked: paraphrase query ===
Query: 'how long do I have to return something'

  #1 [d1] score=3.746  SKU-88421 return policy: items may be returned within 30 days with original receipt for a full refund.
  #2 [d2] score=2.735  If you'd like to send back something you bought, you generally have a month from the purchase date, as long as you kept your proof of purchase.
  #3 [d5] score=1.398  Store credit is issued instead of a refund for returns made after 30 